# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Kadirbek Tileuzhan
**Student ID:** s23068828
**Group:** CSS-4007
**Repository:** https://github.com/Tileuzhan99/css4007-hw3

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> **Tool used:** Claude (Anthropic), through Claude Code.
>
> **What I used it for:** planning how to lay the notebook out; writing and debugging
> the code in Part B and Part C; checking my Part A arithmetic after I had worked it
> out by hand; and drafting and editing the wording of the written answers in
> Part A, B3, C3 and the final questions.
>
> **What it did not do:** every number, chunk id, score, token count and model answer
> quoted anywhere in this notebook is the output of a cell in this notebook, run on my
> own machine with my own API keys. No result here was supplied by the model from
> memory, and no table was typed in by hand without a cell printing it first.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokenisation (README §3, followed exactly):** lowercase everything; split at spaces
**and at hyphens**, so `Retrieval-Augmented` becomes `retrieval`, `augmented`; drop
punctuation; remove no stop words. `|d|` is the number of tokens in the document.

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**, with **N = 3**

**Query terms:** `How does RAG retrieve information?` gives **how, does, rag, retrieve, information** (5 terms)

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln((3 − 0 + 0.5)/(0 + 0.5)) = ln 7 = 1.9459 |
| does | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| information | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |

**BM25 of each document** (the sum for every document, one line per term that contributes):

> Every query term has **f(t, d) = 0 in every document**, so not one term contributes a
> line to any sum. BM25 multiplies idf(t) by f(t, d), and the whole numerator
> `idf(t) · f(t,d) · (k1 + 1)` is zero whenever f(t, d) = 0:
>
> - **score(q, D1)** = 0 (how) + 0 (does) + 0 (rag) + 0 (retrieve) + 0 (information) = **0**
> - **score(q, D2)** = 0 + 0 + 0 + 0 + 0 = **0**
> - **score(q, D3)** = 0 + 0 + 0 + 0 + 0 = **0**
>
> I checked this twice, as README §3 asks, and it survives the check. Term by term:
>
> - `how`, `does`, `information` — these strings do not occur anywhere in D1, D2 or D3.
> - `retrieve` — D2 contains `retrieval` and `retrieves`. Without stemming those are
>   three *different* tokens, and BM25 compares tokens as strings, so f(`retrieve`, D2) = 0.
> - `rag` — D2 is literally about Retrieval-Augmented Generation and contains all three
>   words the acronym is built from, but the token `rag` never appears, so it cannot match.
>
> Note that idf is not zero and is in fact at its maximum here (ln 7 ≈ 1.9459): with
> n(t) = 0 the formula treats every query term as maximally rare. The IDF is large, the
> term frequency is zero, and zero is what multiplies through.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | tie (1–3) |
| D2 | 0.0000 | tie (1–3) |
| D3 | 0.0000 | tie (1–3) |

**There is no ranking in (a): a three-way tie at zero.** A real BM25 index asked this
query would return an empty result set rather than a bad one.

## A1 (b) — BM25 with the stems from README §3

Stems applied wherever the word appears, in the documents and in the query:
`retrieve/retrieves/retrieval` to `retriev`, `generate/generating/generation` to `gener`,
`information` to `inform`, `does` to `doe`. Every other token stays as it is, and the
tokenisation is otherwise identical to (a), so the lengths do not change.

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, **gener**, text, and, answer, questions | 9 |
| D2 | **retriev**, augmented, **gener**, **retriev**, relevant, documents, before, **gener**, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 25 / 3 = **8.3333** (unchanged — stemming replaces tokens, it does not remove them)

**Stemmed query terms:** **how, doe, rag, retriev, inform**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln((3 − 0 + 0.5)/(0 + 0.5)) = ln 7 = 1.9459 |
| doe | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| rag | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |
| **retriev** | 0 | **2** | 0 | **1** | ln((3 − 1 + 0.5)/(1 + 0.5)) = ln(2.5/1.5) = **0.5108** |
| inform | 0 | 0 | 0 | 0 | ln 7 = 1.9459 |

**BM25 of each document**, worked:

> Only **`retriev`** contributes anything, and only to D2. For that one term:
>
> length normalisation for D2: `1 − b + b·|d|/avgdl` = 1 − 0.75 + 0.75 · 10/8.3333
> = 0.25 + 0.75 · 1.2 = 0.25 + 0.9 = **1.1500**
>
> denominator: `f + k1 · norm` = 2 + 1.5 · 1.1500 = 2 + 1.7250 = **3.7250**
>
> numerator: `idf · f · (k1 + 1)` = 0.5108 · 2 · 2.5 = **2.5541**
>
> - **score(q, D2)** = 0 (how) + 0 (doe) + 0 (rag) + **2.5541 / 3.7250** + 0 (inform)
>   = **0.6857**
> - **score(q, D1)** = 0 + 0 + 0 + 0 + 0 = **0** — D1 was stemmed too, and `generate` became
>   `gener`, but `gener` is not a query term, so it never meets the query.
> - **score(q, D3)** = 0 + 0 + 0 + 0 + 0 = **0** — not one token of D3 is touched by the stem list.

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | tie (2–3) |
| D2 | **0.6857** | **1** |
| D3 | 0.0000 | tie (2–3) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> **In (a) it retrieves nothing at all.** All three scores are exactly 0, so there is no
> first document — the result is an empty set, or an arbitrary tie. The reason is that
> BM25 matches *token strings*, and none of `how`, `does`, `rag`, `retrieve`,
> `information` appears as a string in any of the three documents. D2 is clearly the
> document a human would want: it is the one about retrieving documents. BM25 cannot see
> that, because the query says `retrieve` while D2 says `retrieval` and `retrieves`. This
> is the vocabulary-mismatch problem in its purest form: the query and the right answer
> are about the same thing in almost the same words, and the surface forms still differ.
>
> **In (b) it retrieves D2, with 0.6857.** One step of stemming collapses `retrieve`,
> `retrieval` and `retrieves` into the single token `retriev`, and suddenly the query has
> a term in common with D2 — twice, which puts f(t, D2) at 2. Nothing else changed: same
> formula, same k1 and b, same lengths. The entire difference between an empty result and
> the correct answer is the normalisation applied to the tokens before indexing.
>
> Two details are worth naming. First, `rag` still does not match even in (b): stemming
> handles inflection, not acronyms or synonyms, so an acronym gap stays a gap. Second,
> the idf of `retriev` (0.5108) is *much lower* than the idf of the four terms that match
> nothing (1.9459). A term that actually occurs is by definition less rare, so matching is
> rewarded less per occurrence — but a small positive contribution beats a large
> coefficient multiplied by zero every time.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = **0.9644**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = **0.78** | 0.9644 | √0.66 = 0.8124 | 0.78 / (0.9644·0.8124) = 0.78 / 0.7835 = **0.9956** |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = **1.08** | 0.9644 | √1.26 = 1.1225 | 1.08 / (0.9644·1.1225) = 1.08 / 1.0825 = **0.9977** |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = **0.34** | 0.9644 | √0.69 = 0.8307 | 0.34 / (0.9644·0.8307) = 0.34 / 0.8011 = **0.4244** |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | **0.9977** | **1** |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2, at 0.9977.** It is also the document a human reader would pick, since the query
> asks how RAG retrieves information and D2 is the sentence about retrieving documents
> before generating an answer. Note how close D1 is, at 0.9956: the gap between first and
> second place is 0.0021. D2's vector [0.9, 0.3, 0.6] is very nearly the query vector
> [0.8, 0.2, 0.5] scaled up by about 1.15, and cosine ignores magnitude entirely, so the
> two point in almost the same direction. D3 is the only one that is genuinely elsewhere,
> because its largest component is the second one (0.8) while the query's largest is the
> first (0.8).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> | | 1st | 2nd | 3rd |
> |---|---|---|---|
> | **A1 (a)** BM25, no stemming | — | — | — (three-way tie at 0.0000) |
> | **A1 (b)** BM25, stemmed | D2 (0.6857) | D1 and D3 tied at 0.0000 | |
> | **A2** cosine | D2 (0.9977) | D1 (0.9956) | D3 (0.4244) |
>
> Compared with **(a)**: completely different. BM25 (a) produced no ranking at all, while
> cosine confidently ranked all three. They cannot even be said to disagree — (a) has no
> opinion to disagree with.
>
> Compared with **(b)**: the **top of the ranking agrees** — both put D2 first — but the
> rest does not. BM25 (b) leaves D1 and D3 tied at exactly zero and so cannot separate
> "somewhat related" from "unrelated". Cosine separates them clearly: D1 at 0.9956 is
> nearly as good as D2, and D3 at 0.4244 is plainly out. So the two methods agree on the
> answer and disagree on everything below it, which is exactly where they differ in kind:
> **BM25 produces zeros, embeddings produce a graded ordering of everything.**

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> Because they match different things. BM25 matches **token strings**; an embedding matches
> **direction in a vector space** that was trained so that related meanings point the same
> way. In this specific query and these specific documents:
>
> - The query word **`retrieve`** against D2's **`retrieval`** and **`retrieves`**. For
>   BM25 in (a) these are three unrelated strings and the score is 0. An embedding model
>   has seen all three forms in the same contexts in training, so they land in nearly the
>   same place and the similarity stays high whatever the suffix.
> - The query word **`RAG`** against D2's **`Retrieval-Augmented Generation`**. BM25 cannot
>   match this even in (b), because stemming fixes inflection and not abbreviation —
>   `rag` and `retriev` are still different strings. An embedding model that has seen the
>   acronym expanded can place them together, and this is the kind of gap no amount of
>   stemming closes.
> - The query word **`information`** against D2's **`documents`**. Pure synonymy, with no
>   shared characters at all. BM25 scores 0 forever; an embedding sees two words that
>   appear in the same contexts.
> - Conversely, **D1 scores 0.9956 under cosine while BM25 gives it 0**, and D1 has nothing
>   to do with retrieving at all — it is about generating text and answering questions.
>   This is the cost of embeddings: a document that is merely *topically nearby* scores
>   almost as high as the correct one, 0.9956 against 0.9977. BM25 would never make that
>   mistake, because `retriev` simply is not in D1. Blur cuts both ways.
>
> In short: BM25 is exact and brittle, embeddings are fuzzy and forgiving. Here fuzziness
> wins, because the query shares no surface form with its answer. On a query like
> `form GR-305` the exactness would win instead, which is what Part B measures.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [1]:
# Optional arithmetic check for Part A. The working above was done by hand;
# this cell only confirms the numbers. No library computes BM25 here - Part A forbids it.
import math, re

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors.",
}
query = "How does RAG retrieve information?"
k1, b = 1.5, 0.75

def tokenise(s):
    """README section 3: lowercase, split at spaces and hyphens, drop punctuation, keep stop words."""
    s = s.lower().replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)
    return s.split()

STEMS = {
    "retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev",
    "generate": "gener", "generating": "gener", "generation": "gener",
    "information": "inform", "does": "doe",
}
def stem(tokens):
    return [STEMS.get(t, t) for t in tokens]

def bm25_table(label, use_stems):
    D = {k: (stem(tokenise(v)) if use_stems else tokenise(v)) for k, v in docs.items()}
    Q = stem(tokenise(query)) if use_stems else tokenise(query)
    N = len(D)
    avgdl = sum(len(v) for v in D.values()) / N

    print(f"=== {label} ===")
    for k, v in D.items():
        print(f"  {k}: |d|={len(v):2d}  {' '.join(v)}")
    print(f"  N={N}  avgdl={avgdl:.4f}")
    print(f"  query terms = {Q}")
    print(f"  {'term':<12}{'f,D1':>6}{'f,D2':>6}{'f,D3':>6}{'n(t)':>6}{'idf':>10}")

    scores = dict.fromkeys(D, 0.0)
    for t in Q:
        f = {k: v.count(t) for k, v in D.items()}
        n = sum(1 for k in D if f[k] > 0)
        idf = math.log((N - n + 0.5) / (n + 0.5))
        print(f"  {t:<12}{f['D1']:>6}{f['D2']:>6}{f['D3']:>6}{n:>6}{idf:>10.4f}")
        for k, v in D.items():
            if f[k]:
                norm = 1 - b + b * len(v) / avgdl
                contrib = idf * f[k] * (k1 + 1) / (f[k] + k1 * norm)
                scores[k] += contrib
                print(f"      -> {k}: norm={norm:.4f}  den={f[k] + k1 * norm:.4f}  "
                      f"num={idf * f[k] * (k1 + 1):.4f}  contrib={contrib:.4f}")
    for rank, k in enumerate(sorted(scores, key=lambda x: -scores[x]), 1):
        print(f"  rank {rank}: {k} = {scores[k]:.4f}")
    print()
    return scores

bm25_table("A1 (a) as written", use_stems=False)
bm25_table("A1 (b) with stems", use_stems=True)

print("=== A2 cosine ===")
q = [0.8, 0.2, 0.5]
vectors = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
norm_q = math.sqrt(sum(x * x for x in q))
print(f"  ||Query|| = sqrt({sum(x * x for x in q):.2f}) = {norm_q:.4f}")
cosines = {}
for k, v in vectors.items():
    dot = sum(a * c for a, c in zip(q, v))
    norm_v = math.sqrt(sum(x * x for x in v))
    cosines[k] = dot / (norm_q * norm_v)
    print(f"  {k}: dot={dot:.2f}  ||{k}||={norm_v:.4f}  cos={cosines[k]:.4f}")
for rank, k in enumerate(sorted(cosines, key=lambda x: -cosines[x]), 1):
    print(f"  rank {rank}: {k} = {cosines[k]:.4f}")

=== A1 (a) as written ===
  D1: |d|= 9  large language models can generate text and answer questions
  D2: |d|=10  retrieval augmented generation retrieves relevant documents before generating an answer
  D3: |d|= 6  embeddings represent text as numerical vectors
  N=3  avgdl=8.3333
  query terms = ['how', 'does', 'rag', 'retrieve', 'information']
  term          f,D1  f,D2  f,D3  n(t)       idf
  how              0     0     0     0    1.9459
  does             0     0     0     0    1.9459
  rag              0     0     0     0    1.9459
  retrieve         0     0     0     0    1.9459
  information      0     0     0     0    1.9459
  rank 1: D1 = 0.0000
  rank 2: D2 = 0.0000
  rank 3: D3 = 0.0000

=== A1 (b) with stems ===
  D1: |d|= 9  large language models can gener text and answer questions
  D2: |d|=10  retriev augmented gener retriev relevant documents before gener an answer
  D3: |d|= 6  embeddings represent text as numerical vectors
  N=3  avgdl=8.3333
  query terms = ['how'

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
# Part B setup: load the corpus and the fixed queries, and define the helpers
# that both retrievers share. Nothing is retrieved in this cell.
import json
import re
from pathlib import Path

import numpy as np

DATA = Path("data")

corpus = [json.loads(line) for line in DATA.joinpath("corpus.jsonl").open(encoding="utf-8") if line.strip()]
fixed_queries = json.loads(DATA.joinpath("queries.json").read_text(encoding="utf-8"))["queries"]
qa_questions = json.loads(DATA.joinpath("qa.json").read_text(encoding="utf-8"))["questions"]

chunk_ids = [c["id"] for c in corpus]
chunk_texts = [c["text"] for c in corpus]
by_id = {c["id"]: c for c in corpus}

TOP_K = 3


def first_line(chunk_id, width=62):
    """The first line of a chunk, shortened, for the top-3 tables."""
    text = by_id[chunk_id]["text"].strip().split("\n")[0]
    return text if len(text) <= width else text[: width - 3] + "..."


def hit_at_k(result_ids, relevant):
    """hit@k: did any chunk listed in `relevant` make it into the results?"""
    return any(cid in relevant for cid in result_ids)


def show_top(retriever_name, query_id, query_text, results, relevant):
    """Print one top-3 block: rank, chunk id, first line of the chunk, score."""
    ids = [cid for cid, _ in results]
    print(f"{query_id}  [{retriever_name}]  {query_text}")
    print(f"      relevant = {relevant}   hit@{len(results)} = {'YES' if hit_at_k(ids, relevant) else 'NO'}")
    for rank, (cid, score) in enumerate(results, 1):
        star = "*" if cid in relevant else " "
        print(f"   {rank}. {star} {cid}  score={score:7.3f}  [{by_id[cid]['lang']}]  {first_line(cid)}")
    print()


print(f"corpus            : {len(corpus)} chunks")
print(f"  real, KZ-*      : {sum(1 for c in corpus if c['id'].startswith('KZ'))}")
print(f"  handbook, GO-*  : {sum(1 for c in corpus if c['id'].startswith('GO'))}")
print(f"  languages       : {dict((lang, sum(1 for c in corpus if c['lang'] == lang)) for lang in sorted({c['lang'] for c in corpus}))}")
print(f"  chunk length    : min={min(len(t) for t in chunk_texts)}  "
      f"max={max(len(t) for t in chunk_texts)}  "
      f"mean={sum(len(t) for t in chunk_texts) / len(chunk_texts):.0f} characters")
print(f"fixed queries     : {len(fixed_queries)}")
print(f"Part C questions  : {len(qa_questions)}")

corpus            : 74 chunks
  real, KZ-*      : 51
  handbook, GO-*  : 23
  languages       : {'en': 71, 'kk': 1, 'ru': 2}
  chunk length    : min=150  max=1171  mean=561 characters
fixed queries     : 8
Part C questions  : 6


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>

In [3]:
# B1 - BM25 with rank_bm25. The formula is the library's; Part A was where I wrote it myself.
from rank_bm25 import BM25Okapi

TOKEN_RE = re.compile(r"\w+", re.UNICODE)


def tokenize(text):
    r"""Lowercase, then keep every run of word characters. No stemming, no stop word list.

    re.UNICODE matters here: three chunks are in Kazakh and Russian, and \w+ keeps
    Cyrillic as word characters, so GO-14, GO-15 and GO-23 are indexed like the rest.
    """
    return TOKEN_RE.findall(text.lower())


bm25 = BM25Okapi([tokenize(t) for t in chunk_texts], k1=1.5, b=0.75)


def bm25_search(query, k=TOP_K):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(-scores)[:k]
    return [(chunk_ids[i], float(scores[i])) for i in order]


print("tokenisation examples")
for sample in ["What is form GR-305 used for?",
               "Can the grant be sent to my mother's card?",
               "Өтінішті қашанға дейін тапсыру керек?"]:
    print(f"   {sample!r}\n      -> {tokenize(sample)}")
print()
print(f"index: {len(chunk_texts)} documents, "
      f"{sum(len(tokenize(t)) for t in chunk_texts)} tokens, "
      f"avgdl={bm25.avgdl:.2f}, k1=1.5, b=0.75")
print("=" * 78)
print()

bm25_results = {}
for q in fixed_queries:
    bm25_results[q["id"]] = bm25_search(q["text"])
    show_top("BM25", q["id"], q["text"], bm25_results[q["id"]], q["relevant"])

bm25_hits = sum(hit_at_k([c for c, _ in bm25_results[q["id"]]], q["relevant"]) for q in fixed_queries)
print(f"BM25 hit@3 on the eight fixed queries: {bm25_hits}/{len(fixed_queries)}")

tokenisation examples
   'What is form GR-305 used for?'
      -> ['what', 'is', 'form', 'gr', '305', 'used', 'for']
   "Can the grant be sent to my mother's card?"
      -> ['can', 'the', 'grant', 'be', 'sent', 'to', 'my', 'mother', 's', 'card']
   'Өтінішті қашанға дейін тапсыру керек?'
      -> ['өтінішті', 'қашанға', 'дейін', 'тапсыру', 'керек']

index: 74 documents, 6938 tokens, avgdl=93.76, k1=1.5, b=0.75

Q-01  [BM25]  What is form GR-305 used for?
      relevant = ['GO-08', 'GO-12', 'GO-23']   hit@3 = YES
   1. * GO-08  score= 15.825  [en]  Form GR-305 is used to appeal a decision of the Grant Offic...
   2. * GO-12  score=  8.195  [en]  A student who disagrees with a decision may appeal within 1...
   3.   GO-06  score=  8.070  [en]  Form GR-104 is the application form for the need-based stud...

Q-02  [BM25]  My family is poor. Is there financial help for students like me?
      relevant = ['GO-01', 'GO-02', 'GO-03']   hit@3 = NO
   1.   KZ-17  score=  8.726  [en]  The lake c

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
# B2 - EmbeddingGemma through sentence-transformers.
# encode_document and encode_query add the two different instruction prefixes that
# EmbeddingGemma was trained with, so the asymmetry is handled by the library, not by me.
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()  # reads HF_TOKEN out of .env, which is gitignored

embedder = SentenceTransformer("google/embeddinggemma-300m")  # float32: Gemma has no float16 path

chunk_embeddings = embedder.encode_document(chunk_texts)  # the whole corpus, embedded once
print(f"chunk embeddings : {chunk_embeddings.shape}  dtype={chunk_embeddings.dtype}")


def embed_search(query, k=TOP_K):
    query_embedding = embedder.encode_query(query)
    similarities = embedder.similarity(query_embedding, chunk_embeddings)[0]
    order = np.argsort(-similarities.numpy())[:k]
    return [(chunk_ids[i], float(similarities[i])) for i in order]


print("=" * 78)
print()

embed_results = {}
for q in fixed_queries:
    embed_results[q["id"]] = embed_search(q["text"])
    show_top("EmbeddingGemma", q["id"], q["text"], embed_results[q["id"]], q["relevant"])

embed_hits = sum(hit_at_k([c for c, _ in embed_results[q["id"]]], q["relevant"]) for q in fixed_queries)
print(f"EmbeddingGemma hit@3 on the eight fixed queries: {embed_hits}/{len(fixed_queries)}")

C:\Users\madhy\hw3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:  11%|█         | 33/314 [00:00<00:00, 321.85it/s]

Loading weights:  21%|██        | 66/314 [00:00<00:01, 182.23it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 842.69it/s]

chunk embeddings : (74, 768)  dtype=float32

Q-01  [EmbeddingGemma]  What is form GR-305 used for?
      relevant = ['GO-08', 'GO-12', 'GO-23']   hit@3 = YES
   1. * GO-08  score=  0.696  [en]  Form GR-305 is used to appeal a decision of the Grant Offic...
   2. * GO-12  score=  0.461  [en]  A student who disagrees with a decision may appeal within 1...
   3. * GO-23  score=  0.450  [ru]  Если студент не согласен с решением, он может подать апелля...

Q-02  [EmbeddingGemma]  My family is poor. Is there financial help for students like me?
      relevant = ['GO-01', 'GO-02', 'GO-03']   hit@3 = YES
   1. * GO-01  score=  0.445  [en]  The need-based study grant 2026 is run by the Grant Office ...
   2.   GO-10  score=  0.412  [en]  Proof of income must accompany form GR-210. Accepted: a sal...
   3.   GO-20  score=  0.409  [en]  The need-based grant is for full-time undergraduate student...



Q-03  [EmbeddingGemma]  Өтінішті қашанға дейін тапсыру керек?
      relevant = ['GO-09', 'GO-14']   hit@3 = YES
   1. * GO-14  score=  0.399  [kk]  2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанғ...
   2. * GO-09  score=  0.350  [en]  Applications for the 2026 grant open on 1 September 2026 an...
   3.   GO-11  score=  0.301  [en]  If a required document is missing, the Grant Office does no...

Q-04  [EmbeddingGemma]  What is the minimum GPA for the grant?
      relevant = ['GO-02']   hit@3 = YES
   1. * GO-02  score=  0.665  [en]  An applicant qualifies for the 2026 grant when all three co...
   2.   GO-05  score=  0.621  [en]  ARCHIVED — kept for reference only. Under the 2025 scheme t...
   3.   GO-17  score=  0.613  [en]  The grant continues only while the student keeps up their s...

Q-05  [EmbeddingGemma]  When was Astana renamed Nur-Sultan, and why?
      relevant = ['KZ-05']   hit@3 = YES
   1. * KZ-05  score=  0.651  [en]  Astana is the capital city of Kazakhstan. 

Q-06  [EmbeddingGemma]  Where did the Soviet Union test its nuclear weapons?
      relevant = ['KZ-37', 'KZ-38']   hit@3 = YES
   1. * KZ-37  score=  0.596  [en]  The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: С...
   2. * KZ-38  score=  0.510  [en]  The Soviet Union conducted 456 nuclear tests at Semipalatin...
   3.   KZ-10  score=  0.331  [en]  The Baikonur Cosmodrome is a spaceport operated by Russia w...

Q-07  [EmbeddingGemma]  Can the grant be sent to my mother's card?
      relevant = ['GO-15']   hit@3 = YES
   1. * GO-15  score=  0.422  [ru]  Грант выплачивается двумя равными частями — в октябре и в ф...
   2.   GO-18  score=  0.417  [en]  A student who goes on academic leave keeps their grant deci...
   3.   GO-01  score=  0.411  [en]  The need-based study grant 2026 is run by the Grant Office ...

Q-08  [EmbeddingGemma]  If my grades drop in the autumn, do I still get paid in February?
      relevant = ['GO-17']   hit@3 = YES
   1. * GO-17  score=  0.559  [en]  Th

## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | `GR-210 income declaration proof documents` | `GO-07`, `GO-10` | **BM25.** The query is built around a form code, and `GR-210` tokenises to `gr` + `210`, where `210` is a rare token that occurs in exactly the four chunks that discuss that form. Exact-token matching should put them straight at the top. Embeddings should struggle instead, because the handbook has **three forms with near-identical names** — GR-104 (application), GR-210 (income declaration) and GR-305 (appeals) — and all three chunks read almost identically: "Form GR-xxx is used to …". A 300M embedding model is likely to treat the three as nearly the same point in space and blur the digits, so I expect at least one wrong form in its top 3. |
| Q-10 | `I want to take a year off from university. Will I lose the money?` | `GO-18` | **Embeddings.** The query and its answer share almost no vocabulary. GO-18 is titled *Academic leave and transfers* and says a student who goes on **academic leave** keeps the decision but **payments stop**; my query says **take a year off** and **lose the money**. Not one content word overlaps: no "academic", no "leave", no "payment". BM25 can only match function words plus possibly `university`, which is frequent across the corpus and so carries almost no IDF, so I expect it to drift to some unrelated chunk. An embedding model should recognise "take a year off" and "academic leave" as the same idea. |

Both predictions are written down above **before** the next cell runs. The results, including any prediction that turns out wrong, are reported as they came out.

In [5]:
# My own queries, with the `relevant` ids decided before looking at any result.
my_queries = [
    {"id": "Q-09",
     "text": "GR-210 income declaration proof documents",
     "relevant": ["GO-07", "GO-10"],
     "expected": "BM25"},
    {"id": "Q-10",
     "text": "I want to take a year off from university. Will I lose the money?",
     "relevant": ["GO-18"],
     "expected": "EmbeddingGemma"},
]

for q in my_queries:
    bm25_results[q["id"]] = bm25_search(q["text"])
    embed_results[q["id"]] = embed_search(q["text"])
    print(f"### {q['id']}  expected winner: {q['expected']}")
    show_top("BM25", q["id"], q["text"], bm25_results[q["id"]], q["relevant"])
    show_top("EmbeddingGemma", q["id"], q["text"], embed_results[q["id"]], q["relevant"])

    bm25_hit = hit_at_k([c for c, _ in bm25_results[q["id"]]], q["relevant"])
    embed_hit = hit_at_k([c for c, _ in embed_results[q["id"]]], q["relevant"])
    if bm25_hit == embed_hit:
        verdict = "tie (both hit)" if bm25_hit else "tie (both missed)"
    else:
        verdict = f"{'BM25' if bm25_hit else 'EmbeddingGemma'} won"
    print(f"   -> outcome: {verdict};  prediction was {q['expected']}  "
          f"({'CORRECT' if verdict.startswith(q['expected']) else 'see the note in B3'})")
    print("=" * 78)
    print()

### Q-09  expected winner: BM25
Q-09  [BM25]  GR-210 income declaration proof documents
      relevant = ['GO-07', 'GO-10']   hit@3 = YES
   1. * GO-07  score= 14.451  [en]  Form GR-210 is the household income declaration. It lists e...
   2.   GO-13  score= 14.318  [en]  A student whose household income changes during the year mu...
   3. * GO-10  score= 12.102  [en]  Proof of income must accompany form GR-210. Accepted: a sal...

Q-09  [EmbeddingGemma]  GR-210 income declaration proof documents
      relevant = ['GO-07', 'GO-10']   hit@3 = YES
   1. * GO-10  score=  0.699  [en]  Proof of income must accompany form GR-210. Accepted: a sal...
   2. * GO-07  score=  0.633  [en]  Form GR-210 is the household income declaration. It lists e...
   3.   GO-13  score=  0.547  [en]  A student whose household income changes during the year mu...

   -> outcome: tie (both hit);  prediction was BM25  (see the note in B3)

### Q-10  expected winner: EmbeddingGemma
Q-10  [BM25]  I want to take a ye

## Results for all ten or more queries: top 3 from each retriever

In [6]:
# All ten queries, both retrievers, side by side. This cell prints the B3 table.
all_queries = list(fixed_queries) + list(my_queries)

header = f"| {'Query':6} | {'BM25 top 3 (ids)':26} | {'hit':3} | {'Embedding top 3 (ids)':26} | {'hit':3} |"
print(header)
print("|" + "-" * 8 + "|" + "-" * 28 + "|" + "-" * 5 + "|" + "-" * 28 + "|" + "-" * 5 + "|")

bm25_total = embed_total = 0
for q in all_queries:
    b_ids = [cid for cid, _ in bm25_results[q["id"]]]
    e_ids = [cid for cid, _ in embed_results[q["id"]]]
    b_hit = hit_at_k(b_ids, q["relevant"])
    e_hit = hit_at_k(e_ids, q["relevant"])
    bm25_total += b_hit
    embed_total += e_hit
    print(f"| {q['id']:6} | {', '.join(b_ids):26} | {'YES' if b_hit else 'no':3} | "
          f"{', '.join(e_ids):26} | {'YES' if e_hit else 'no':3} |")

n = len(all_queries)
print(f"| {'TOTAL':6} | {'':26} | {bm25_total}/{n} | {'':26} | {embed_total}/{n} |")
print()
print(f"BM25           hit@3 = {bm25_total}/{n}  ({bm25_total / n:.0%})")
print(f"EmbeddingGemma hit@3 = {embed_total}/{n}  ({embed_total / n:.0%})")
print()

print("where they differ")
print("-" * 78)
for q in all_queries:
    b_ids = [cid for cid, _ in bm25_results[q["id"]]]
    e_ids = [cid for cid, _ in embed_results[q["id"]]]
    b_hit, e_hit = hit_at_k(b_ids, q["relevant"]), hit_at_k(e_ids, q["relevant"])
    if b_hit != e_hit:
        print(f"  {q['id']}: only {'BM25' if b_hit else 'EmbeddingGemma'} hit   "
              f"| BM25 {b_ids} | emb {e_ids} | relevant {q['relevant']}")
    elif b_ids == e_ids:
        print(f"  {q['id']}: identical top 3 {b_ids}")
print()
print("scores of the top hit, for the B3 write-up")
print("-" * 78)
for q in all_queries:
    b = bm25_results[q["id"]][0]
    e = embed_results[q["id"]][0]
    print(f"  {q['id']}: BM25 best {b[0]} {b[1]:.3f}   |   emb best {e[0]} {e[1]:.4f}")

# ---------------------------------------------------------------------------
# A focused probe for B3. The handbook has three forms with near-identical
# names, and my Q-09 prediction was that the embedding model would blur them.
# This checks that claim directly: each form code on its own, no other words,
# scored against the three chunks that define the three forms.
FORM_CHUNKS = {"GO-06": "GR-104, application",
               "GO-07": "GR-210, income declaration",
               "GO-08": "GR-305, appeals"}
CODE_TO_CHUNK = {"GR-104": "GO-06", "GR-210": "GO-07", "GR-305": "GO-08"}

print()
print("probe: can either retriever tell the three forms apart?")
print("=" * 78)
for code, right in CODE_TO_CHUNK.items():
    b_scores = bm25.get_scores(tokenize(code))
    e_scores = embedder.similarity(embedder.encode_query(code), chunk_embeddings)[0].numpy()
    b_by_id = {cid: float(b_scores[i]) for i, cid in enumerate(chunk_ids)}
    e_by_id = {cid: float(e_scores[i]) for i, cid in enumerate(chunk_ids)}

    print(f"\nquery {code!r}   tokenises to {tokenize(code)}   correct chunk = {right}")
    for cid, label in FORM_CHUNKS.items():
        mark = "<-- correct" if cid == right else ""
        print(f"   {cid} ({label:28}) BM25={b_by_id[cid]:7.3f}   emb={e_by_id[cid]:.4f}  {mark}")

    wrong = [cid for cid in FORM_CHUNKS if cid != right]
    b_ratio = b_by_id[right] / max(b_by_id[w] for w in wrong)
    e_ratio = e_by_id[right] / max(e_by_id[w] for w in wrong)
    print(f"   correct form over the best wrong form:  BM25 {b_ratio:.2f}x   emb {e_ratio:.2f}x")
    print(f"   top 3  BM25: {[(cid, round(b_by_id[cid], 3)) for cid in sorted(b_by_id, key=lambda c: -b_by_id[c])[:3]]}")
    print(f"   top 3  emb : {[(cid, round(e_by_id[cid], 4)) for cid in sorted(e_by_id, key=lambda c: -e_by_id[c])[:3]]}")

| Query  | BM25 top 3 (ids)           | hit | Embedding top 3 (ids)      | hit |
|--------|----------------------------|-----|----------------------------|-----|
| Q-01   | GO-08, GO-12, GO-06        | YES | GO-08, GO-12, GO-23        | YES |
| Q-02   | KZ-17, GO-20, KZ-25        | no  | GO-01, GO-10, GO-20        | YES |
| Q-03   | GO-14, KZ-01, KZ-03        | YES | GO-14, GO-09, GO-11        | YES |
| Q-04   | GO-05, GO-17, GO-19        | no  | GO-02, GO-05, GO-17        | YES |
| Q-05   | KZ-05, KZ-06, KZ-11        | YES | KZ-05, KZ-06, KZ-03        | YES |
| Q-06   | KZ-37, KZ-38, KZ-03        | YES | KZ-37, KZ-38, KZ-10        | YES |
| Q-07   | GO-21, GO-16, GO-01        | no  | GO-15, GO-18, GO-01        | YES |
| Q-08   | GO-17, GO-05, GO-04        | YES | GO-17, GO-18, GO-15        | YES |
| Q-09   | GO-07, GO-13, GO-10        | YES | GO-10, GO-07, GO-13        | YES |
| Q-10   | GO-18, GO-16, KZ-07        | YES | GO-18, GO-17, GO-01        | YES |
| TOTAL  |                  


query 'GR-305'   tokenises to ['gr', '305']   correct chunk = GO-08
   GO-06 (GR-104, application         ) BM25=  2.840   emb=0.3402  
   GO-07 (GR-210, income declaration  ) BM25=  2.531   emb=0.3737  
   GO-08 (GR-305, appeals             ) BM25=  7.417   emb=0.6000  <-- correct
   correct form over the best wrong form:  BM25 2.61x   emb 1.61x
   top 3  BM25: [('GO-08', 7.417), ('GO-23', 5.965), ('GO-12', 5.005)]
   top 3  emb : [('GO-08', 0.6), ('GO-12', 0.4183), ('GO-23', 0.3963)]


## B3 — comparison

Every number in this table is printed by the cell above. Retrieval is deterministic here,
and these figures reproduced exactly across three full runs of the notebook.

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-03 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-07, GO-13, GO-10 | ✓ | GO-10, GO-07, GO-13 | ✓ |
| Q-10 | GO-18, GO-16, KZ-07 | ✓ | GO-18, GO-17, GO-01 | ✓ |
| **hit@3 total** | | **7/10** | | **10/10** |

**First, the two predictions I wrote down before running Q-09 and Q-10. Both were wrong,
and the way they were wrong is the useful part.**

> **Q-09** — I predicted BM25 would win because the embedding model would blur three
> forms with near-identical names. **It did not blur them.** Both retrievers hit, and the
> probe in the cell above tests the claim directly: given the bare code `GR-305`,
> EmbeddingGemma puts GO-08 first at 0.6000 against 0.3737 and 0.3402 for the other two
> form chunks; `GR-104` puts GO-06 first at 0.5102; `GR-210` puts GO-07 first at 0.5736.
> Three codes, three correct answers, no confusion. My mental model of "a 300M embedding
> model cannot keep three digit strings apart" was simply false, and the probe is what
> showed it.
>
> **Q-10** — I predicted embeddings would win, and they hit; but BM25 hit too, so there
> was no win. What the scores show, though, is that BM25 arrived there by luck:
> GO-18 **8.910**, GO-16 **8.767**, KZ-07 **8.687**. The gap between the right answer and
> the runner-up is 0.14 on a scale where its confident hits score 15 to 19, and third place
> went to **KZ-07, a chunk about Almaty being a UNESCO Creative City of Music**. BM25 had
> no idea; it returned a nearly flat list and the correct chunk happened to be at the top of
> it. EmbeddingGemma separated GO-18 at 0.5502 from 0.4600 and 0.3690 — a real decision.
>
> The lesson is about the metric, not the retrievers: **hit@3 is coarse**, and it scores a
> confident hit and an accidental one identically.

**Both retrievers agree:** which query, and why was it easy?

> **Q-06, "Where did the Soviet Union test its nuclear weapons?"** Both retrievers return
> KZ-37 then KZ-38, in that order, and both are confident: BM25 **19.090** and **17.168**,
> its two highest scores in the whole run; EmbeddingGemma **0.5960** and **0.5102**.
>
> It was easy because **the lexical and the semantic signal point the same way**. The query
> words `soviet`, `union`, `test`, `nuclear` appear literally in those chunks — KZ-38 opens
> "The Soviet Union conducted 456 nuclear tests at Semipalatinsk" — so BM25 matches on
> strings, and the chunks are also *about* that topic and nothing else, so the embedding
> matches on meaning. There is no vocabulary gap to cross and no distractor: the corpus has
> exactly two chunks on nuclear testing and they are the two that are relevant. When the
> words of the question are also the words of the answer, the choice of retriever does not
> matter.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> **There is no such query in my runs, and I am not going to invent one.** BM25 scored
> 7/10 against 10/10, and all three disagreements (Q-02, Q-04, Q-07) went the other way.
> On this corpus BM25 never beat EmbeddingGemma on hit@3 — not once.
>
> What BM25 *did* do better is **separate an exact match from everything near it**, and the
> probe above measures it. Taking the ratio of the correct form chunk's score to the best
> wrong form chunk's score:
>
> | Bare query | BM25, correct ÷ best wrong | EmbeddingGemma, correct ÷ best wrong |
> |---|---:|---:|
> | `GR-104` | **2.71×** | 1.33× |
> | `GR-210` | **2.42×** | 1.56× |
> | `GR-305` | **2.61×** | 1.61× |
>
> Both rank correctly; BM25 is roughly twice as decisive about it. The exact string doing
> the work is the digit group, because my tokeniser splits `GR-305` into `gr` + `305`, and
> `305` occurs in only three of the 74 chunks, so its IDF is high. `gr` occurs in twelve and
> contributes almost nothing. On Q-01 this produces the sharpest margin of the whole run:
> GO-08 at **15.825** against **8.195** for second place, a 1.93× gap, where the embedding
> model's margin is 1.51×.
>
> So the honest answer is: BM25 is better *calibrated* on literal identifiers, and on a
> corpus of 74 short chunks that advantage never converted into a single extra hit@3. It
> would start to matter on a corpus large enough that the top 3 is a real contest, or on
> queries that are keyword lookups rather than sentences. Eight of my ten queries are
> written as natural-language questions, which is the condition embeddings are built for.

**Embeddings did better:** which query, and what could BM25 not match?

> Three queries, and each fails for a different reason. All three are in the table above.
>
> **Q-07, "Can the grant be sent to my mother's card?" — another language.** This is the
> clearest case. The answer is GO-15, which is **in Russian**: *"Счёт должен быть открыт на
> имя самого студента: перевод на счёт родителей или других лиц невозможен."* The query is
> in English. After tokenisation the query gives `can, the, grant, be, sent, to, my, mother,
> s, card` and GO-15 gives Cyrillic tokens. The intersection is **empty**, so BM25 cannot
> score GO-15 above zero on any content word; it returned GO-21 (**14.652**), the chunk
> about the office being in room 214, because that one repeats `grant` and `office` in
> English. EmbeddingGemma put GO-15 first at **0.4216** — it was trained multilingually, so
> "mother's card" and "счёт родителей" land near each other and the alphabet is irrelevant.
>
> **Q-02, "My family is poor. Is there financial help for students like me?" — paraphrase.**
> The handbook never uses the word `poor`, and it never uses `financial help`; it says
> `low-income household` and `need-based study grant`. Nothing in the query matches anything
> in GO-01, GO-02 or GO-03, so BM25 scored on the only words left — and `like`, `family`,
> `students` picked out **KZ-17, a chunk about Lake Balkhash shrinking (8.726)**, with
> **KZ-25 on how beshbarmak is cooked** in third. A retriever answering a question about
> poverty with a lake and a meat dish is the vocabulary-mismatch problem of Part A, at
> scale. EmbeddingGemma returned GO-01 first (**0.4446**).
>
> **Q-04, "What is the minimum GPA for the grant?" — the dangerous one.** BM25 returned
> **GO-05 first, at 12.780**, and GO-02 not at all. Read GO-05's title: *"Archived: the 2025
> scheme (no longer in force)"*. It outranks the current rule because it is the chunk that
> contains the literal string `minimum GPA` — GO-02 phrases the same rule as "their
> cumulative GPA is at least 2.67", which contains neither `minimum` nor `GPA for the grant`.
> So BM25 did not merely miss; **it put the superseded answer in first place, worded exactly
> the way a student would ask for it.** EmbeddingGemma put GO-02 first (**0.6652**) and
> GO-05 second (**0.6213**) — it found the right chunk, but it also brought the trap along.
> What happens next is a Part C question, and C3 follows it through.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [7]:
# Part C setup: the model client, the prompt builder and the token counter.
# Nothing is sent in this cell.
import os

import tiktoken
from openai import OpenAI

load_dotenv()  # OPENAI_API_KEY comes out of .env, which is gitignored
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is missing from .env"

MODEL = "gpt-5.6-luna"
client = OpenAI()
ENC = tiktoken.get_encoding("o200k_base")

# The system message from README section 5, used unchanged for both RAG systems.
SYSTEM_PROMPT = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)


def n_tokens(text):
    """Token count under o200k_base, the encoding gpt-5.6-luna uses."""
    return len(ENC.encode(text))


def build_context(results):
    """The numbered context blocks, exactly as they go into the user message."""
    blocks = []
    for i, (cid, _score) in enumerate(results, 1):
        blocks.append(f"[{i}] ({cid}) {by_id[cid]['title']}\n{by_id[cid]['text'].strip()}")
    return "\n\n".join(blocks)


def build_messages(question, results=None):
    """RAG prompt when `results` is given; the bare question when it is not."""
    if results is None:
        # LLM only: no system message, no context. Just the question.
        return [{"role": "user", "content": question}]
    user = f"Context:\n\n{build_context(results)}\n\nQuestion: {question}"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]


def ask(question, results=None):
    """Send one question and return (answer_text, usage)."""
    messages = build_messages(question, results)
    response = client.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content.strip(), response.usage


print(f"model            : {MODEL}")
print(f"encoding         : o200k_base")
print(f"OPENAI_API_KEY   : loaded from .env ({len(os.environ['OPENAI_API_KEY'])} characters, not printed)")
print(f"system prompt    : {n_tokens(SYSTEM_PROMPT)} tokens, {len(SYSTEM_PROMPT)} characters")

model            : gpt-5.6-luna
encoding         : o200k_base
OPENAI_API_KEY   : loaded from .env (164 characters, not printed)
system prompt    : 44 tokens, 178 characters


## C1 — the complete prompt, printed before it is sent

In [8]:
# C1 - one question, retrieved with EmbeddingGemma, printed in full before it is sent.
C1_QUESTION = next(q for q in qa_questions if q["id"] == "C-01")
c1_results = embed_search(C1_QUESTION["question"], k=TOP_K)

print(f"question : {C1_QUESTION['id']}  {C1_QUESTION['question']}")
print(f"reference: {C1_QUESTION['reference']}")
print(f"relevant : {C1_QUESTION['relevant']}")
print(f"retrieved: {[(cid, round(s, 4)) for cid, s in c1_results]}")
print()

c1_messages = build_messages(C1_QUESTION["question"], c1_results)

print("=" * 78)
print("THE COMPLETE PROMPT, AS THE MODEL RECEIVES IT")
print("=" * 78)
for message in c1_messages:
    print(f"\n----- role: {message['role']} -----")
    print(message["content"])
print("\n" + "=" * 78)
print()

# --- the token account, counted with tiktoken o200k_base -------------------
question_text = f"Question: {C1_QUESTION['question']}"
context_text = build_context(c1_results)

tok_system = n_tokens(SYSTEM_PROMPT)
tok_question = n_tokens(question_text)
tok_context = n_tokens(context_text)
tok_sum = tok_system + tok_question + tok_context

print(f"{'Part of the input':<24}{'Tokens':>8}")
print("-" * 32)
print(f"{'System instructions':<24}{tok_system:>8}")
print(f"{'Question':<24}{tok_question:>8}")
print(f"{'Retrieved context':<24}{tok_context:>8}")
print(f"{'TOTAL (tiktoken)':<24}{tok_sum:>8}")
print()

answer, usage = ask(C1_QUESTION["question"], c1_results)

print(f"{'usage.prompt_tokens':<24}{usage.prompt_tokens:>8}   <- what the API actually billed")
print(f"{'difference':<24}{usage.prompt_tokens - tok_sum:>+8}")
print(f"{'completion_tokens':<24}{usage.completion_tokens:>8}")
print()
print("--- per-chunk breakdown of the retrieved context ---")
for i, (cid, score) in enumerate(c1_results, 1):
    block = f"[{i}] ({cid}) {by_id[cid]['title']}\n{by_id[cid]['text'].strip()}"
    print(f"  [{i}] {cid}  lang={by_id[cid]['lang']}  score={score:.4f}  "
          f"{n_tokens(block):>4} tokens  {len(block):>5} characters")
print(f"  chunks retrieved      : {len(c1_results)}")
print(f"  context total         : {tok_context} tokens / {len(context_text)} characters")
print(f"  mean tokens per chunk : {tok_context / len(c1_results):.1f}")
print()
print("--- the answer ---")
print(answer)

question : C-01  How much does a student in income band 2 receive, and when is it paid?
reference: 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant : ['GO-04', 'GO-15']
retrieved: [('GO-04', 0.5845), ('GO-03', 0.4842), ('GO-15', 0.4506)]

THE COMPLETE PROMPT, AS THE MODEL RECEIVES IT

----- role: system -----
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

----- role: user -----
Context:

[1] (GO-04) How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] (G

usage.prompt_tokens          393   <- what the API actually billed
difference                   +12
completion_tokens             53

--- per-chunk breakdown of the retrieved context ---
  [1] GO-04  lang=en  score=0.5845   104 tokens    423 characters
  [2] GO-03  lang=en  score=0.4842   115 tokens    433 characters
  [3] GO-15  lang=ru  score=0.4506    98 tokens    361 characters
  chunks retrieved      : 3
  context total         : 317 tokens / 1221 characters
  mean tokens per chunk : 105.7

--- the answer ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 20 |
| Retrieved context | 317 |
| **Total input** | **381** |
| **`usage.prompt_tokens` from the response** | **393** |

- Number of retrieved chunks: **3** — GO-04 `[en]`, GO-03 `[en]`, GO-15 `[ru]`
- Tokens per chunk: **104**, **115**, **98** — mean **105.7**
- Total context size (tokens / characters): **317 tokens / 1221 characters**

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> The gap is **+12 tokens**, and it is not noise — it splits into two exact parts.
>
> **3 tokens are prompt text I did not count.** `build_messages` assembles the user message
> as `f"Context:\n\n{context}\n\nQuestion: {question}"`. My three table rows measured the
> system prompt, the string `"Question: …"` and the context blocks, but not the literal
> `"Context:\n\n"` header (**2 tokens**) or the `"\n\n"` that separates the context from the
> question (**1 token**). I counted the contents of the envelope, not the envelope.
>
> **The remaining 9 tokens are the chat format itself.** tiktoken encodes a *string*. The
> API does not receive strings; it receives a list of role-tagged messages, and the server
> serialises them with special tokens that mark where each message starts, which role it
> belongs to, and where it ends, plus the opening of the assistant turn the model is to
> continue. Two messages here, so roughly four or five such tokens per message. They are
> real input tokens and they are billed, but they exist only after the message list has been
> flattened, so no amount of counting the text can find them.
>
> 381 + 3 + 9 = 393. This is the same gap as in HW1: a tokeniser run over your own text is
> a lower bound on what a chat endpoint charges, because the protocol adds its own
> scaffolding that you never wrote.

**What exactly does retrieval add to the model's input?**

> Literally: **1221 characters of text, pasted into the user message, as 317 tokens.**
> Nothing else. No tool, no lookup, no reference, no capability.
>
> Being precise about it, retrieval added three string blocks, each prefixed with a number
> and the chunk's id and title, joined by blank lines, between the line `Context:` and the
> line `Question: …`. One of them (GO-15) is in Russian, and the model receives it as
> Russian text with no translation step and no marker that it is a different language.
>
> The model's situation is unchanged in kind by all this. It is still continuing a prompt.
> Before retrieval the prompt was 76 tokens and the model had to answer from its weights,
> which is why the LLM-only system replied to this question by asking which country I meant.
> After retrieval the prompt is 393 tokens and the answer is sitting inside it, so
> continuing the prompt plausibly means copying 150,000 and October and February out of
> block [1]. The retrieval step did not teach the model anything and did not give it access
> to anything — **it changed the text the model was asked to continue**, and that is the
> whole mechanism.
>
> One concrete consequence, visible in the breakdown above: GO-03 is in the context at 115
> tokens, the largest of the three, and it is not relevant to this question — it defines the
> income bands rather than the amounts. 36% of what retrieval added was irrelevant, the
> model paid for it, and the answer cites only `[1]`.

## C2 — three systems, six questions

In [9]:
# C2 - six questions through three systems. Top 3 for both RAG systems.
c2 = {}

for q in qa_questions:
    question, qid = q["question"], q["id"]
    b_results = bm25_search(question, k=TOP_K)
    e_results = embed_search(question, k=TOP_K)

    llm_answer, llm_usage = ask(question)                 # system 1: no context
    bm25_answer, bm25_usage = ask(question, b_results)    # system 2: BM25 RAG
    emb_answer, emb_usage = ask(question, e_results)      # system 3: embedding RAG

    c2[qid] = {
        "question": question,
        "reference": q["reference"],
        "relevant": q["relevant"],
        "bm25_results": b_results,
        "embed_results": e_results,
        "llm_only": llm_answer,
        "bm25_rag": bm25_answer,
        "embed_rag": emb_answer,
        "prompt_tokens": {"llm_only": llm_usage.prompt_tokens,
                          "bm25_rag": bm25_usage.prompt_tokens,
                          "embed_rag": emb_usage.prompt_tokens},
    }

    print("=" * 78)
    print(f"{qid}  {question}")
    print(f"REFERENCE : {q['reference']}")
    print(f"relevant  : {q['relevant']}")
    print(f"BM25 top3 : {[(cid, round(s, 2)) for cid, s in b_results]}  "
          f"hit@3={'YES' if hit_at_k([c for c, _ in b_results], q['relevant']) else 'NO'}")
    print(f"emb  top3 : {[(cid, round(s, 4)) for cid, s in e_results]}  "
          f"hit@3={'YES' if hit_at_k([c for c, _ in e_results], q['relevant']) else 'NO'}")
    print()
    print(f"--- 1. LLM ONLY ({llm_usage.prompt_tokens} prompt tokens) ---")
    print(llm_answer)
    print()
    print(f"--- 2. BM25 RAG ({bm25_usage.prompt_tokens} prompt tokens) ---")
    print(bm25_answer)
    print()
    print(f"--- 3. EMBEDDING RAG ({emb_usage.prompt_tokens} prompt tokens) ---")
    print(emb_answer)
    print()

C-01  How much does a student in income band 2 receive, and when is it paid?
REFERENCE : 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant  : ['GO-04', 'GO-15']
BM25 top3 : [('GO-04', 20.22), ('GO-02', 18.74), ('GO-01', 17.24)]  hit@3=YES
emb  top3 : [('GO-04', 0.5845), ('GO-03', 0.4842), ('GO-15', 0.4506)]  hit@3=YES

--- 1. LLM ONLY (24 prompt tokens) ---
Which student funding scheme and country do you mean? “Income band 2” varies by program, and the payment amount and schedule depend on the academic year.

--- 2. BM25 RAG (401 prompt tokens) ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

--- 3. EMBEDDING RAG (393 prompt tokens) ---
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**—the first in **October** and the second in **February**

C-02  My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE : 14 calendar days from the date on the decision letter, using form GR-305.
relevant  : ['GO-08', 'GO-12', 'GO-23']
BM25 top3 : [('GO-05', 17.24), ('GO-08', 14.53), ('GO-12', 13.57)]  hit@3=YES
emb  top3 : [('GO-08', 0.6029), ('GO-12', 0.5591), ('GO-11', 0.5343)]  hit@3=YES

--- 1. LLM ONLY (27 prompt tokens) ---
The deadline and form depend on the **grant program and jurisdiction**. They should be stated in the refusal/decision letter—often under “appeal,” “review,” or “reconsideration.”

Please tell me:

- The **grant name or funding body**
- Your **country/state/province**
- The **date on the refusal letter** (and, if shown, the appeal deadline)

If the letter does not specify a deadline or form, contact the funding body immediately and ask for the appeal/review procedure in writing. Don’t assume that an email or a general complaints form will preserve your appeal rights.

--- 2.

C-03  What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE : A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant  : ['GO-02']
BM25 top3 : [('GO-16', 15.8), ('GO-02', 14.06), ('GO-06', 13.85)]  hit@3=YES
emb  top3 : [('GO-02', 0.6286), ('GO-20', 0.5691), ('GO-05', 0.5451)]  hit@3=YES

--- 1. LLM ONLY (20 prompt tokens) ---
It depends on the specific need-based grant. Many grants require maintaining **Satisfactory Academic Progress**, often around a **2.0 GPA**, but some programs have different requirements. For example, the federal Pell Grant does **not** have a specific GPA requirement to qualify initially, though you must meet your school’s progress standards to continue receiving it.

Which grant or school are you asking about?

--- 2. BM25 RAG (371 prompt tokens) ---
The minimum GPA is **2.67 on a 4.0 scale**. [2]

--- 3. EMBEDDING RAG (361 prompt tokens) ---
The minimum GPA needed is **2.67 on a 4.0 sca

C-04  When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE : A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
relevant  : ['KZ-43']
BM25 top3 : [('KZ-43', 25.97), ('KZ-14', 24.06), ('KZ-06', 16.84)]  hit@3=YES
emb  top3 : [('KZ-14', 0.7204), ('KZ-43', 0.6927), ('KZ-05', 0.4342)]  hit@3=YES

--- 1. LLM ONLY (32 prompt tokens) ---
Kazakhstan formally decided to replace the Cyrillic Kazakh alphabet with a Latin-based one on **26 October 2017**, under President Nursultan Nazarbayev.

The original target was to complete the transition by **2025**, but that timetable was extended. The current plan is for the switch to be completed by **2031**.

--- 2. BM25 RAG (626 prompt tokens) ---
Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition was initially due by 2025, but the deadline was later extended, and it is now due

C-05  When was Narxoz University founded, and what was it called at the start?
REFERENCE : In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.
relevant  : ['KZ-44']
BM25 top3 : [('KZ-44', 17.57), ('KZ-11', 13.76), ('KZ-07', 12.26)]  hit@3=YES
emb  top3 : [('KZ-44', 0.6603), ('KZ-05', 0.2715), ('KZ-14', 0.2508)]  hit@3=YES

--- 1. LLM ONLY (23 prompt tokens) ---
Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy**.

--- 2. BM25 RAG (538 prompt tokens) ---
Narxoz University was founded in **1963** as the **Almaty Institute of National Economy**. [1]

--- 3. EMBEDDING RAG (633 prompt tokens) ---
Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]



C-06  I am a master's student. How much need-based grant can I get?
REFERENCE : The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
relevant  : ['GO-20']
BM25 top3 : [('GO-16', 17.0), ('GO-01', 14.17), ('GO-20', 13.72)]  hit@3=YES
emb  top3 : [('GO-20', 0.6664), ('GO-16', 0.6161), ('GO-01', 0.5539)]  hit@3=YES

--- 1. LLM ONLY (21 prompt tokens) ---
It depends mainly on your **country, school, program, income, assets, enrollment status, and dependency/family information**.

If you mean the **United States**:

- **Federal Pell Grant:** Generally **not available for master’s students**; it is primarily for undergraduate students who have not earned a bachelor’s degree.
- **Federal Supplemental Educational Opportunity Grant (FSEOG):** Usually also limited to undergraduates.
- **TEACH Grant:** Master’s students in eligible teaching programs may receive up to the federal annual limit—currently **up to $4,000 per year**, subject t

Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

Marked by hand against `reference`, the way Kazakh was marked in HW1: a correct answer in
different words counts, a right number with an invented form code does not.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ no answer — "which country or scheme do you mean?" | ✓ 150,000 tenge, October + February | ✓ 150,000 tenge, two instalments of 75,000, October + February |
| C-02 | ✗ no answer — "check the refusal letter", generic advice | ✓ 14 calendar days, form GR-305 | ✓ 14 calendar days, form GR-305 |
| C-03 | ✗ answered about US Pell Grants, "commonly 2.0–3.0" | ✓ 2.67 on the 4.0 scale | ✓ 2.67 on the 4.0 scale |
| C-04 | ✓ 26 October 2017 decree, 2025 target, now 2031 | ✓ October 2017, originally 2025, postponed to 2031 | ✓ October 2017, due to finish 2031 (omits the 2025 target) |
| C-05 | ✓ 1963, Alma-Ata Institute of National Economy | ✓ 1963, Almaty Institute of National Economy | ✓ 1963, Almaty Institute of National Economy |
| C-06 | ✗ invented a US answer — Pell, TEACH "$4,000", "$20,500" loans | ✓ "I don't know the amount", separate scheme, ask the department | ✓ "I do not know the amount", separate scheme, ask the department |
| **agrees with `reference`** | **2/6** | **6/6** | **6/6** |

Notes on three of the markings, since they are judgement calls:

- **C-01, Embedding RAG** says the instalments are **75,000 tenge** each. That number is not
  in the context. It is 150,000 ÷ 2, and GO-04 states "two equal instalments", so the
  arithmetic is entailed by what was sent rather than invented. Marked ✓. Had it produced a
  form code or a date that was not in the context, that would have been the other case.
- **C-05, LLM only** says **Alma-Ata** Institute of National Economy where the reference says
  **Almaty**. Same institution, older transliteration of the same city name. A correct answer
  in different words is correct, so ✓.
- **C-04, Embedding RAG** is marked ✓ because the question asks two things — when the decision
  was taken and when the switch is now due to finish — and it answers both, October 2017 and
  2031. It is nonetheless the **weakest** of the three answers to that question, and it is one
  of my three C3 cases below, because the 2025 target and the January 2021 decision were both
  sitting in its context and did not reach the answer.

The shape of the table is the result: **retrieval moved the score from 2/6 to 6/6**, and the
two questions the LLM answered unaided (C-04, C-05) are the two drawn from English Wikipedia,
which it had read in training. Every question about the fictional grant handbook — C-01,
C-02, C-03, C-06 — it got wrong without context and right with it, from either retriever.

## C3 — question 1 of 3: C-03

*"What is the minimum GPA needed to qualify for the need-based grant?"*
Reference: **a cumulative GPA of at least 2.67 on the 4.0 scale.** The 3.0 in GO-05 is the
archived 2025 rule. `relevant = ["GO-02"]`. Everything below is printed by the cell under it.

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. GO-16 — **15.80** — *Combining the grant with other support*
> 2. **GO-02** — **14.06** — *Who qualifies* ← the relevant chunk
> 3. GO-06 — **13.85** — *Form GR-104: the application form*
>
> hit@3 = **yes**, at rank 2. Note what is *not* here: **GO-05, the archived rule, did not
> make BM25's top 3 for this wording.** That is pure luck and the contrast is instructive.
> The same fact asked as Q-04 in Part B — "What is the minimum GPA for the grant?" — gave
> BM25 `GO-05 (12.780), GO-17, GO-19` and **no GO-02 at all**. One phrasing retrieves the
> current rule, the other retrieves only the superseded one. The difference between the two
> queries is the words `needed to qualify`, which happen to overlap GO-02's title *Who
> qualifies*.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. **GO-02** — **0.6286** — *Who qualifies* ← the relevant chunk, first
> 2. GO-20 — **0.5691** — *Who cannot apply*
> 3. **GO-05** — **0.5451** — *Archived: the 2025 scheme (no longer in force)*
>
> hit@3 = **yes**, at rank 1. And here the trap **is** in the context: the embedding model
> ranked the archived chunk third, because semantically it is about exactly this topic — a
> minimum GPA for this grant. Being out of force is not a property an embedding can see.

**3. What was sent to the model** (the context blocks):

> **BM25 RAG** sent GO-16, GO-02, GO-06 — 371 prompt tokens. The only GPA figure anywhere in
> that context is GO-02's: *"their cumulative GPA is at least 2.67 on the 4.0 scale"*.
>
> **Embedding RAG** sent GO-02, GO-20, GO-05 — 361 prompt tokens. This context contains
> **two** GPA figures that contradict each other: 2.67 in block [1] and, in block [3],
> *"Under the 2025 scheme the minimum GPA for the grant was 3.0"*. Block [3] also carries its
> own warning label, which turns out to be the decisive detail: it opens *"ARCHIVED — kept for
> reference only"* and closes *"These rules were replaced on 1 June 2026 and do not apply to
> any application made for the 2026 academic year."*

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **More accurate, decisively.** Without context the model answered a different question
> entirely: it explained US federal aid — *"Federal Pell Grants generally don't require a
> minimum GPA … State and college grants may set their own requirements, commonly 2.0–3.0"* —
> and asked which school I meant. Not wrong about America, and worth nothing here, because the
> handbook is fictional and 2.67 is not a number the model could guess.
>
> With context, both RAG systems answered **2.67 on a 4.0 scale**, and both cited the block
> holding it: BM25 RAG cited `[2]`, embedding RAG cited `[1]`. The citations match where GO-02
> actually sat in each prompt, which is the check worth doing — a right number with a wrong
> citation would mean the model was reciting rather than reading.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> **Nothing was missed, and the interesting result is the failure that did not happen.**
>
> Embedding RAG was handed both the current rule and the archived one, and still answered
> 2.67. By the test in README §5 this cannot be a generation problem — the fact was in the
> context and the answer has it. Generation did more than not fail: it **used the metadata in
> the chunk**. GO-05 says in its own text that it is archived and does not apply to 2026
> applications, the system prompt says to answer from the context, and the model resolved the
> contradiction the way the context told it to. That only worked because whoever wrote that
> chunk put the warning *inside the text*. An embedding cannot read "no longer in force", but
> a language model reading the retrieved text can.
>
> **The real failure in this pair lives in Part B, and it is a retrieval problem.** On Q-04,
> BM25's top 3 was `GO-05, GO-17, GO-19`. GO-02 is absent, so the string `2.67` never entered
> the context, and the only GPA available to copy would have been **3.0** — with no
> contradiction for the model to notice and no way for it to know anything was wrong. By the
> same test: the fact was not in the context, so retrieval failed. The answer would have been
> wrong, confidently, and traceably not the model's fault. The margin between a correct RAG
> answer and a plausible wrong one on this question is entirely in the retriever.

In [10]:
# C3, question 1 of 3 - everything that was retrieved and everything that was sent.
QID = "C-03"
case = c2[QID]

print("=" * 78)
print(f"{QID}  {case['question']}")
print(f"REFERENCE: {case['reference']}")
print(f"relevant : {case['relevant']}")
print("=" * 78)

print("\n1. WHAT BM25 RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["bm25_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:7.3f}  [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['bm25_results']], case['relevant'])}")

print("\n2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["embed_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:.4f}   [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['embed_results']], case['relevant'])}")

print("\n3. WHAT WAS SENT TO THE MODEL")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    context = build_context(results)
    print(f"\n   ===== context blocks sent by {label} ({n_tokens(context)} tokens) =====")
    for line in context.split("\n"):
        print(f"   {line}")

print("\n4. THE THREE ANSWERS")
for label, key in [("LLM ONLY", "llm_only"), ("BM25 RAG", "bm25_rag"), ("EMBEDDING RAG", "embed_rag")]:
    print(f"\n   --- {label} ---")
    for line in case[key].split("\n"):
        print(f"   {line}")

print("\n5. IS THE FACT IN THE CONTEXT AT ALL?")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    sent_ids = [cid for cid, _ in results]
    present = [cid for cid in case["relevant"] if cid in sent_ids]
    verdict = ("the relevant chunk WAS in the context -> any miss is a GENERATION problem"
               if present else
               "no relevant chunk reached the context -> a miss is a RETRIEVAL problem")
    print(f"   {label:14} sent={sent_ids}  relevant present={present or 'none'}")
    print(f"   {'':14} {verdict}")

C-03  What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant : ['GO-02']

1. WHAT BM25 RETRIEVED (top 3)
   1.   GO-16  score= 15.799  [en]  Combining the grant with other support
   2. * GO-02  score= 14.059  [en]  Who qualifies
   3.   GO-06  score= 13.852  [en]  Form GR-104: the application form
   hit@3 = True

2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)
   1. * GO-02  score=0.6286   [en]  Who qualifies
   2.   GO-20  score=0.5691   [en]  Who cannot apply
   3.   GO-05  score=0.5451   [en]  Archived: the 2025 scheme (no longer in force)
   hit@3 = True

3. WHAT WAS SENT TO THE MODEL

   ===== context blocks sent by BM25 RAG (299 tokens) =====
   [1] (GO-16) Combining the grant with other support
   The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all univ

## C3 — question 2 of 3: C-06

*"I am a master's student. How much need-based grant can I get?"*
Reference: **the corpus does not say.** GO-20 states that master's students have a separate
scheme the handbook does not describe, so the only correct response from a RAG system is that
it does not know. `relevant = ["GO-20"]`.

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. GO-16 — **17.00** — *Combining the grant with other support*
> 2. GO-01 — **14.17** — *About the need-based study grant 2026*
> 3. **GO-20** — **13.72** — *Who cannot apply* ← the relevant chunk
>
> hit@3 = **yes**, at rank 3, and only just. `master` appears once in the whole corpus, in
> GO-20, so the one token that actually decides this question carries high IDF — but GO-16 and
> GO-01 win on repeated matches of `grant`, `need`, `based` and `student`, which appear in
> nearly every GO chunk and are worth little each. BM25 ranked quantity of common terms above
> the single rare term that mattered.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. **GO-20** — **0.6664** — *Who cannot apply* ← the relevant chunk, first
> 2. GO-16 — **0.6161** — *Combining the grant with other support*
> 3. GO-01 — **0.5539** — *About the need-based study grant 2026*
>
> hit@3 = **yes**, at rank 1. Same three chunks as BM25, exactly reversed. "I am a master's
> student" is a statement about eligibility, and GO-20 is the chunk about who is not eligible;
> the embedding caught that relation where term frequency did not.

**3. What was sent to the model** (the context blocks):

> Both systems sent the **same three chunks**, in different orders, at **333 prompt tokens**
> each. So this question isolates the model's behaviour almost perfectly: the context is held
> constant and only the ordering changes.
>
> The decisive sentence, in both prompts, is in GO-20: *"Master's and doctoral students are
> supported by a separate scheme, which this handbook does not describe; they should ask their
> own department for its rules."*
>
> Note what the context does **not** contain: any grant amount for a master's student. It
> contains amounts — GO-16 caps total support at 400,000 tenge, GO-01 describes the
> undergraduate grant — and a model inclined to be helpful has plenty of plausible numbers
> within reach.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **More grounded, and this is the sharpest contrast in the whole assignment.** This is the
> one question where the right answer is a refusal, and it is the one question the LLM-only
> system failed most expensively.
>
> **LLM only** produced a long, formatted, confident briefing on American graduate aid:
> *"Federal Pell Grant: generally not available for regular master's programs … TEACH Grant:
> up to roughly $4,000 per year … Federal Direct Unsubsidized Loan: usually up to $20,500 per
> academic year"*, and closed by telling me to file a FAFSA. Every figure is irrelevant to a
> Kazakh university handbook, none is marked as a guess, and the answer never says it does not
> know which scheme I am asking about. If a student read that, they would act on it.
>
> **BM25 RAG:** *"I don't know the amount. Master's students are supported by a separate
> scheme, and this handbook does not describe its rules; you should ask your department."* `[3]`
>
> **Embedding RAG:** *"I do not know the amount. Master's students are not covered by this
> need-based grant and should ask their department about its separate scheme."* `[1]`
>
> Both are correct, both cite the block that actually holds GO-20 in their own prompt, and both
> answered a question whose answer is absent **without filling the absence in**. Retrieval did
> not supply a fact here. It supplied the boundary of what is known, and the instruction to
> stay inside the context turned that boundary into an honest answer.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> **Nothing was missed by either RAG system.** GO-20 reached both contexts, both answers
> contain what GO-20 says, and both decline to invent the amount. Neither a retrieval nor a
> generation failure.
>
> **The LLM-only answer is a generation failure of the purest kind**, and it is worth naming
> as such even though no retrieval was involved. The fact "this handbook does not cover
> master's students" was not in its input, so it could not have been copied — but the model did
> not report an absence, it substituted a different, better-documented world (US federal aid)
> and answered confidently about that one instead. The $20,500 figure is real, which is what
> makes it dangerous: it is not a hallucinated number, it is a correct number about the wrong
> country, delivered without a hedge.
>
> That is the asymmetry the whole assignment is built to show. Retrieval's contribution to
> C-06 is not a fact — the corpus has none to give. It is that **the model was told what it is
> allowed to answer from**, so "the context does not contain this" became an available
> response. Without a context there is no such thing as a fact being absent from it, and a
> model with nothing to ground against has no way to distinguish "I was not given this" from
> "this is not knowable", so it reaches for whatever it does know.

In [11]:
# C3, question 2 of 3 - everything that was retrieved and everything that was sent.
QID = "C-06"
case = c2[QID]

print("=" * 78)
print(f"{QID}  {case['question']}")
print(f"REFERENCE: {case['reference']}")
print(f"relevant : {case['relevant']}")
print("=" * 78)

print("\n1. WHAT BM25 RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["bm25_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:7.3f}  [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['bm25_results']], case['relevant'])}")

print("\n2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["embed_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:.4f}   [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['embed_results']], case['relevant'])}")

print("\n3. WHAT WAS SENT TO THE MODEL")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    context = build_context(results)
    print(f"\n   ===== context blocks sent by {label} ({n_tokens(context)} tokens) =====")
    for line in context.split("\n"):
        print(f"   {line}")

print("\n4. THE THREE ANSWERS")
for label, key in [("LLM ONLY", "llm_only"), ("BM25 RAG", "bm25_rag"), ("EMBEDDING RAG", "embed_rag")]:
    print(f"\n   --- {label} ---")
    for line in case[key].split("\n"):
        print(f"   {line}")

print("\n5. IS THE FACT IN THE CONTEXT AT ALL?")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    sent_ids = [cid for cid, _ in results]
    present = [cid for cid in case["relevant"] if cid in sent_ids]
    verdict = ("the relevant chunk WAS in the context -> any miss is a GENERATION problem"
               if present else
               "no relevant chunk reached the context -> a miss is a RETRIEVAL problem")
    print(f"   {label:14} sent={sent_ids}  relevant present={present or 'none'}")
    print(f"   {'':14} {verdict}")

C-06  I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
relevant : ['GO-20']

1. WHAT BM25 RETRIEVED (top 3)
   1.   GO-16  score= 16.996  [en]  Combining the grant with other support
   2.   GO-01  score= 14.173  [en]  About the need-based study grant 2026
   3. * GO-20  score= 13.720  [en]  Who cannot apply
   hit@3 = True

2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)
   1. * GO-20  score=0.6664   [en]  Who cannot apply
   2.   GO-16  score=0.6161   [en]  Combining the grant with other support
   3.   GO-01  score=0.5539   [en]  About the need-based study grant 2026
   hit@3 = True

3. WHAT WAS SENT TO THE MODEL

   ===== context blocks sent by BM25 RAG (260 tokens) =====
   [1] (GO-16) Combining the grant with other support
   The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any ot

## C3 — question 3 of 3: C-04

*"When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is
the switch now due to finish?"*
Reference: **a presidential decree of October 2017 ordered the change for 2025; in January 2021
the target was moved to 2031.** `relevant = ["KZ-43"]`.

**This is my "worse than expected" case.** It is the only question in C2 where a RAG system
produced a *less complete* answer than the model with no context at all, and the reason is not
retrieval.

**1. What BM25 retrieved** (top 3, ids and scores):

> 1. **KZ-43** — **25.97** — *Kazakh alphabets* ← the relevant chunk, first
> 2. KZ-14 — **24.06** — *Kazakh language (part 3)*
> 3. KZ-06 — **16.84** — *Almaty (part 1)*
>
> hit@3 = **yes**, at rank 1, and 25.97 is the highest BM25 score in the entire notebook. The
> query is unusually friendly to lexical matching: `kazakh`, `alphabet`, `cyrillic`, `latin`,
> `switch` are all low-frequency in this corpus and all appear literally in KZ-43.

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> 1. KZ-14 — **0.7204** — *Kazakh language (part 3)*
> 2. **KZ-43** — **0.6927** — *Kazakh alphabets* ← the relevant chunk, second
> 3. KZ-05 — **0.4342** — *Astana*
>
> hit@3 = **yes**, at rank 2. The two retrievers **disagree about which of the two chunks is
> first**, and they disagree in the direction that matters: the embedding model preferred
> KZ-14, which is *about the Kazakh language in general*, over KZ-43, which is the chunk that
> actually contains both dates. KZ-14 is topically broader and mentions January 2021 — but
> about the alphabet's letter inventory, not about the deadline. This is embedding blur doing
> real damage for once: close in meaning, wrong on specifics.

**3. What was sent to the model** (the context blocks):

> **BM25 RAG** sent KZ-43, KZ-14, KZ-06 — 626 prompt tokens.
> **Embedding RAG** sent KZ-14, KZ-43, KZ-05 — 605 prompt tokens.
>
> Both contexts contain **KZ-43 in full**, and KZ-43 contains the complete answer in two
> consecutive sentences: *"In October 2017, a presidential decree in Kazakhstan ordered a
> transition from the Cyrillic to Latin script to be implemented by 2025. In January 2021, the
> target year for finishing the transition was pushed back to 2031."*
>
> Both dates, both decisions, one chunk, in both prompts. Whatever happens next cannot be
> blamed on the retriever.

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> **It made one answer no better and the other one worse — in both cases, less specific than
> no retrieval at all.** All three systems are marked ✓, and the ranking by quality runs
> backwards:
>
> **LLM only** (32 prompt tokens, no context): *"Kazakhstan made the decision in 2017:
> President Nursultan Nazarbayev announced the move in April, and signed the formal decree on
> 26 October 2017. The original target was 2025 … now scheduled to be completed by 2031."*
> The most detailed of the three — it has the exact date of the decree, the name of the
> president, the original 2025 target and the current 2031 one.
>
> **BM25 RAG** (626 tokens): *"Kazakhstan decided to switch from Cyrillic to Latin in October
> 2017. The completion deadline, originally 2025, was later postponed to 2031."* `[1][2]`
> Complete against the reference, and still less specific than the free answer.
>
> **Embedding RAG** (605 tokens): *"Kazakhstan decided to switch from Cyrillic to Latin in
> October 2017. The transition is now due to finish by 2031."* `[1][2]`
> **The 2025 target and the January 2021 reversal are both gone**, although both were in
> block [2] of its own prompt. It answers the question as literally asked and drops the
> chronology that makes the answer mean something.
>
> So retrieval cost 573 extra prompt tokens here and bought nothing. This is the control case
> the corpus was designed to provide: KZ-* chunks are real English Wikipedia, which the model
> read in training, so **retrieval adds nothing where the model already knows the material**.
> It is not free either — it narrows the answer to what was retrieved.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> **A generation problem, unambiguously, in the embedding RAG answer.**
>
> Apply the test from README §5 directly. The missing facts are the 2025 target and the
> January 2021 decision to move it to 2031. Were they in the context? Yes — block [2] of the
> embedding RAG prompt is KZ-43, which says *"…to be implemented by 2025. In January 2021, the
> target year for finishing the transition was pushed back to 2031."* The fact was in the
> context and the answer does not have it, so **generation failed**, not retrieval. Retrieval
> did its job: hit@3 yes, the relevant chunk present, the full answer sitting in the prompt.
>
> Why it failed is visible in the two prompts. The sentence the model needed is in the
> **second** block for the embedding system and the **first** block for BM25, and the BM25
> answer is the complete one. The same model, the same instruction, the same chunk — reordered
> — and the detail survives in one answer and not the other. Rank inside the context is not
> neutral; it behaves like emphasis. I would not claim that one comparison proves it, but it
> is the only variable that differs here.
>
> There is also a **retrieval** observation worth separating out, even though it did not cause
> the miss: EmbeddingGemma ranked KZ-14 above KZ-43 (0.7204 against 0.6927). KZ-14 is the chunk
> that mentions January 2021 *about the alphabet's letters* and says the switch is "scheduled
> to be phased in from 2023 to 2031" — close enough to the question to win on cosine, and not
> the chunk that answers it. Had K been 1, embedding RAG would have received KZ-14 alone and
> could not have given the 2017 decree date at all. BM25 at K = 1 would have been fine. The
> retriever that scored 10/10 on hit@3 would have been the one to fail at K = 1 on this
> question, which is a reminder that hit@3 hides rank.

In [12]:
# C3, question 3 of 3 - everything that was retrieved and everything that was sent.
QID = "C-04"
case = c2[QID]

print("=" * 78)
print(f"{QID}  {case['question']}")
print(f"REFERENCE: {case['reference']}")
print(f"relevant : {case['relevant']}")
print("=" * 78)

print("\n1. WHAT BM25 RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["bm25_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:7.3f}  [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['bm25_results']], case['relevant'])}")

print("\n2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)")
for rank, (cid, score) in enumerate(case["embed_results"], 1):
    star = "*" if cid in case["relevant"] else " "
    print(f"   {rank}. {star} {cid}  score={score:.4f}   [{by_id[cid]['lang']}]  {by_id[cid]['title']}")
print(f"   hit@3 = {hit_at_k([c for c, _ in case['embed_results']], case['relevant'])}")

print("\n3. WHAT WAS SENT TO THE MODEL")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    context = build_context(results)
    print(f"\n   ===== context blocks sent by {label} ({n_tokens(context)} tokens) =====")
    for line in context.split("\n"):
        print(f"   {line}")

print("\n4. THE THREE ANSWERS")
for label, key in [("LLM ONLY", "llm_only"), ("BM25 RAG", "bm25_rag"), ("EMBEDDING RAG", "embed_rag")]:
    print(f"\n   --- {label} ---")
    for line in case[key].split("\n"):
        print(f"   {line}")

print("\n5. IS THE FACT IN THE CONTEXT AT ALL?")
for label, results in [("BM25 RAG", case["bm25_results"]), ("EMBEDDING RAG", case["embed_results"])]:
    sent_ids = [cid for cid, _ in results]
    present = [cid for cid in case["relevant"] if cid in sent_ids]
    verdict = ("the relevant chunk WAS in the context -> any miss is a GENERATION problem"
               if present else
               "no relevant chunk reached the context -> a miss is a RETRIEVAL problem")
    print(f"   {label:14} sent={sent_ids}  relevant present={present or 'none'}")
    print(f"   {'':14} {verdict}")

C-04  When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
REFERENCE: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
relevant : ['KZ-43']

1. WHAT BM25 RETRIEVED (top 3)
   1. * KZ-43  score= 25.966  [en]  Kazakh alphabets
   2.   KZ-14  score= 24.062  [en]  Kazakh language (part 3)
   3.   KZ-06  score= 16.835  [en]  Almaty (part 1)
   hit@3 = True

2. WHAT EMBEDDINGGEMMA RETRIEVED (top 3)
   1.   KZ-14  score=0.7204   [en]  Kazakh language (part 3)
   2. * KZ-43  score=0.6927   [en]  Kazakh alphabets
   3.   KZ-05  score=0.4342   [en]  Astana
   hit@3 = True

3. WHAT WAS SENT TO THE MODEL

   ===== context blocks sent by BM25 RAG (542 tokens) =====
   [1] (KZ-43) Kazakh alphabets
   The Kazakh language has been written mainly in four scripts at various points of time—Old Turkic, Cyrillic, Latin, and Arabic—each having a distinct alphabet. The Arabic 

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> What they compare. BM25 compares **token strings** and scores a document by how often the
> query's exact terms occur in it, weighted by how rare each term is in the collection; a term
> that is not present contributes exactly nothing. Embedding retrieval compares **positions in
> a vector space** — both the query and the chunk become one 768-dimensional vector each in my
> run, and relevance is the angle between them, so text that means the same thing scores high
> even with no word in common. The practical consequence shows in Part A: with no stemming,
> BM25 scored all three documents **0.0000** for a query whose answer was sitting in D2, while
> cosine ranked D2 first at **0.9977**. BM25 is exact and can return nothing; embeddings are
> approximate and always return a ranking.

**2. When can BM25 perform better than embeddings?**

> When the query *is* the string you need: identifiers, codes, names, numbers, quoted phrases.
> My probe in Part B measures this on the three near-identical form names. Given the bare code
> `GR-104`, BM25 scores the correct chunk **2.71×** the best wrong form; EmbeddingGemma, which
> also ranks it first, scores it only **1.33×** higher. BM25 is about twice as decisive,
> because `104` occurs in few chunks and so carries high IDF, whereas to the embedding model
> all three chunks read as "a sentence about a grant office form". BM25 also needs no model, no
> GPU and no 1.2 GB download, and it adds nothing to latency.
>
> Honestly, though: **on this corpus that advantage never produced a single extra hit@3.**
> BM25 finished 7/10 against 10/10 and lost all three disagreements. 74 short chunks and
> mostly sentence-shaped queries is the setting embeddings are best in.

**3. When can embeddings perform better than BM25?**

> Whenever the query and the answer do not share surface forms. My three disagreements are one
> example of each failure mode:
>
> - **Another language** — Q-07 asked about a "mother's card" in English; the answer, GO-15, is
>   in Russian. Token overlap is zero, so BM25 could not score it at all and returned GO-21
>   (14.652) about room 214. EmbeddingGemma ranked GO-15 first at 0.4216.
> - **Paraphrase** — Q-02 said "my family is poor"; the handbook says "low-income household".
>   BM25 returned **KZ-17, about Lake Balkhash shrinking** (8.726) and KZ-25 about cooking
>   beshbarmak. Embeddings returned GO-01 first.
> - **Different wording of the same rule** — Q-04 asked for the "minimum GPA"; GO-02 phrases it
>   as "cumulative GPA is at least 2.67" and never uses the word "minimum", so BM25 ranked the
>   **archived** GO-05 first instead, which does use it.

**4. Why is cosine similarity used with embeddings?**

> Because it measures direction and ignores length, and in an embedding space direction is what
> carries meaning while length mostly carries incidental things like how long the text is. The
> arithmetic is visible in A2: D2 = [0.9, 0.3, 0.6] is almost exactly the query [0.8, 0.2, 0.5]
> multiplied by 1.15. A dot product would reward D2 for being longer; cosine divides that out
> and reports 0.9977, so a two-sentence chunk and a six-sentence chunk saying the same thing
> score the same. It is also bounded and cheap — for unit-normalised vectors it is just a dot
> product, which is why embedding all 74 chunks and scoring every query against them took no
> index at all.

**5. What information is actually sent to an LLM in a RAG system?**

> Text, and nothing but text. For C-01 the model received a 44-token system message, then one
> user message containing the word `Context:`, three numbered blocks totalling **317 tokens /
> 1221 characters** carrying the chunks' ids, titles and full text, and then the line
> `Question: …`. **393 prompt tokens in total.** No document, no link, no database handle, no
> search tool. The chunk ids travel as part of the text, which is the only reason the model can
> cite `[1]` — it is copying a label it was shown, not referring to a record it can look up.

**6. Does the LLM search the original documents itself?**

> No. `corpus.jsonl` never leaves my machine, and the model has no access to it. My code
> searches — `BM25Okapi.get_scores` or a cosine over the embedding matrix — my code sorts, my
> code takes the top 3, and my code pastes those three strings into the prompt. The model's
> only input is the finished prompt. If my retriever returns the wrong chunk, the model has no
> way to notice and no second attempt: on Q-04 BM25 handed back the archived GO-05, and a model
> given that top 3 would have reported 3.0 with no indication that anything was missing.

**7. What happens if retrieval returns an irrelevant chunk?**

> Three things, and my runs show all three. **It costs tokens**: in C-01, GO-03 took 115 of the
> 317 context tokens — 36% of what retrieval added — and the answer cites only `[1]`. **It can
> be harmless**: the model simply ignored GO-03, GO-16 and GO-06 where they were not needed, so
> precision is not the whole story. **It can be actively dangerous when the irrelevant chunk is
> plausible and contradicts the right one**: embedding RAG's context for C-03 contained both
> 2.67 (GO-02) and 3.0 (GO-05, archived), and the answer came out correct only because GO-05
> says *inside its own text* "ARCHIVED — kept for reference only … do not apply to any
> application made for the 2026 academic year". Strip that sentence and the same retrieval
> becomes a coin flip. A retrieved chunk is an instruction to the model to treat that text as
> true.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> Raising K trades recall for precision. More chunks means the right one is likelier to be in
> there at all, and also that more wrong ones are, plus a longer prompt to pay for and more
> competing text for the answer to be drawn from.
>
> I ran C-03 at K = 1, 3 and 8 with EmbeddingGemma (cell below):
>
> | K | chunks sent | context tokens | prompt tokens | GO-05 (archived) present? | answer |
> |---:|---|---:|---:|:---:|---|
> | 1 | GO-02 | 113 | 185 | no | **2.67** ✓ |
> | 3 | GO-02, GO-20, GO-05 | 289 | 361 | **yes** | **2.67** ✓ |
> | 8 | GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17 | 784 | 856 | **yes** | **2.67** ✓ |
>
> **The answer did not change.** K = 8 cost **4.6× the prompt tokens of K = 1** for an
> identical result, and that is the finding: on this question the extra context was pure
> overhead. It also did not do the harm I expected — GO-05 enters the context at K = 3 and
> stays, and the model never reported 3.0, for the reason in question 7: GO-05 labels itself
> archived.
>
> Where K would matter here is **below** 3, on a different question. On C-04, EmbeddingGemma
> ranked KZ-14 above KZ-43, so at K = 1 the embedding system would have received the chunk that
> does *not* contain the 2017 decree date and could not have answered at all; BM25 at K = 1
> would have been fine, because it ranked KZ-43 first at 25.97. So the honest summary from my
> own runs: raising K past the point where the answer is already present buys nothing and costs
> linearly, while lowering it below that point is where retrieval starts failing outright — and
> that point depends on the retriever, not just on K.

In [13]:
# Final question 8 - Top-K on one question. C-03 is the right one to try it on,
# because the corpus holds both the current rule (GO-02) and an archived one (GO-05).
TOPK_QID = "C-03"
topk_question = next(q for q in qa_questions if q["id"] == TOPK_QID)

print(f"{TOPK_QID}  {topk_question['question']}")
print(f"REFERENCE: {topk_question['reference']}")
print(f"relevant : {topk_question['relevant']}")
print()

for k in (1, 3, 8):
    results = embed_search(topk_question["question"], k=k)
    context = build_context(results)
    answer, usage = ask(topk_question["question"], results)
    ids = [cid for cid, _ in results]

    print("=" * 78)
    print(f"K = {k}")
    print(f"   chunks sent   : {ids}")
    print(f"   relevant in it: {[cid for cid in topk_question['relevant'] if cid in ids] or 'none'}")
    print(f"   GO-05 present : {'YES - the archived 2025 rule is in the context' if 'GO-05' in ids else 'no'}")
    print(f"   context tokens: {n_tokens(context)}   prompt_tokens: {usage.prompt_tokens}")
    print(f"   answer        : {answer}")
    print()

C-03  What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant : ['GO-02']



K = 1
   chunks sent   : ['GO-02']
   relevant in it: ['GO-02']
   GO-05 present : no
   context tokens: 113   prompt_tokens: 185
   answer        : The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



K = 3
   chunks sent   : ['GO-02', 'GO-20', 'GO-05']
   relevant in it: ['GO-02']
   GO-05 present : YES - the archived 2025 rule is in the context
   context tokens: 289   prompt_tokens: 361
   answer        : The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



K = 8
   chunks sent   : ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']
   relevant in it: ['GO-02']
   GO-05 present : YES - the archived 2025 rule is in the context
   context tokens: 784   prompt_tokens: 856
   answer        : The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**. [1]



**9. Is a vector database required to build a RAG system?**

> No, and this notebook is the evidence. The vectors live in **one numpy array of shape
> (74, 768), float32** — about 227 KB — held in a variable called `chunk_embeddings`. Search is
> `embedder.similarity(query_vector, chunk_embeddings)` followed by `np.argsort`, which is a
> brute-force scan of all 74 rows and takes microseconds. There is no index, no server, no
> persistence layer, and nothing was installed for it.
>
> A vector database earns its place at a scale this corpus does not reach: when a full scan
> becomes too slow and approximate nearest-neighbour search is needed, when the vectors no
> longer fit in memory, when they must persist across processes or be shared between machines,
> or when metadata filtering and incremental updates are required. For 74 chunks all of that is
> overhead. The thing a RAG system actually needs is a function from a query to the top K
> chunks; a database is one implementation of it, and `argsort` is another.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

> **EmbeddingGemma, clearly, on retrieval — and the two were indistinguishable once a model was
> attached.**
>
> | Measure | BM25 | EmbeddingGemma |
> |---|:---:|:---:|
> | hit@3 over my ten queries | **7/10** | **10/10** |
> | C2 agreement with `reference`, as a RAG system | **6/6** | **6/6** |
> | (LLM only, for reference) | | **2/6** |
>
> The retrieval evidence is the three disagreements, all of which went to embeddings: **Q-02**
> (BM25 answered a question about poverty with Lake Balkhash at 8.726), **Q-04** (BM25 ranked
> the archived 2025 rule first at 12.780 and missed GO-02 entirely) and **Q-07** (the answer is
> a Russian chunk, GO-15, which BM25 cannot reach from an English query; it returned GO-21 at
> 14.652 instead). Embeddings hit every one of the ten, including the Kazakh query Q-03, where
> they found both relevant chunks — GO-14 at 0.3992 and the English GO-09 at 0.3500 — while
> BM25 found only GO-14 and scored literally 0.000 for everything in second and third place.
>
> Two things stop this from being a blanket verdict. **First, the C2 column is a tie at 6/6.**
> Three retrieval misses did not become three wrong answers, because the C2 questions happen to
> be phrased so that BM25 reaches a relevant chunk — C-03 is the clean illustration: asked as
> Q-04 it loses GO-02, asked as C-03 with the words "needed to qualify" it finds it at rank 2.
> The gap between the retrievers is therefore real but narrower in effect than 7/10 vs 10/10
> suggests, and it would widen on a corpus where the answer is not almost always in the top
> handful. **Second, hit@3 flatters embeddings on Q-10**, where BM25 technically hit with GO-18
> at 8.910 against 8.767 and 8.687 — effectively a tie with a chunk about Almaty's music scene.
> A metric that records that as a success is not measuring confidence.
>
> On the evidence I have: **embeddings for retrieval on this corpus, with BM25 better
> calibrated on literal identifiers** (2.4–2.7× margin on bare form codes against 1.3–1.6×),
> which is exactly the complementarity that makes hybrid retrieval the standard answer.

---
# Conclusion

> The number I did not expect is **2/6 against 6/6**. Both RAG systems answered every question
> correctly and the same model with no context answered two, and both of those two — C-04 on the
> alphabet reform, C-05 on Narxoz — come from English Wikipedia, which the model read in
> training. Every question about the fictional grant handbook it got wrong unaided and right
> with context. Retrieval added exactly what was not already in the weights, and nothing else:
> on C-04 it cost 573 extra prompt tokens and made the embedding answer *worse*, dropping the
> 2025 target that was sitting in its own block [2].
>
> C-06 is the one I would put in front of anyone who thinks RAG is about accuracy. Asked how
> much grant a master's student gets — a question the corpus deliberately does not answer — the
> unaided model produced a confident page on US federal aid, Pell and TEACH grants, "$20,500
> per academic year", and told me to file a FAFSA. Real numbers, wrong country, no hedge. Both
> RAG systems said they did not know and pointed at GO-20. What retrieval supplied there was
> not a fact; it was **the boundary of what was known**, and a prompt that made "this is not in
> the context" something the model could say.
>
> **What I would build next, in order.** First, **hybrid retrieval**: BM25 and embeddings are
> good at opposite things in a way I can now quantify — embeddings 10/10 on hit@3 but only
> 1.3–1.6× margin on form codes, BM25 7/10 but 2.4–2.7× on the same codes — so I would score
> with both, fuse the ranks, and expect to keep the paraphrase and cross-language wins without
> giving up the precision on identifiers. Second, **a freshness signal outside the text**. The
> C-03 trap was defused only because GO-05 happens to say "ARCHIVED" inside its own body; an
> embedding cannot see that a rule is out of force, and I would not want a correct answer to
> depend on a document having been written carefully. A `status` or `valid_until` field in the
> chunk metadata, filtered before ranking, makes that structural instead of lucky. Third,
> **a metric that is not hit@3**. It gave BM25 full marks on Q-10 for a 0.14-point accident and
> told me nothing about rank, which is what actually decided C-04 — the same chunk in position
> [1] versus position [2] produced a complete answer and an incomplete one. MRR plus a
> report of where in the context the answer landed would have surfaced that.
>
> The thing I will keep from this assignment is smaller than the pipeline and harder to unsee:
> the model never opened a document. Every correct detail in a RAG answer arrived as characters
> I chose to paste into a prompt, which means **every RAG failure is attributable** — either the
> chunk was not in the context, and that is mine to fix in the retriever, or it was in the
> context and was not used, and that is the prompt's or the model's. C-04 is the second kind and
> Q-04 is the first, and I could only tell them apart because I printed what was sent.